# Data Wrangling: Duplicates, Missing Values and Compensation

Part of my IBM Data Analyst Capstone project. In this notebook I prepare the developer survey data for analysis: remove duplicate rows, handle missing values in `EdLevel`, and prepare the `ConvertedCompYearly` compensation column.

The file used here contains 10 exact duplicate rows (the earlier duplicates notebook used a different version of the file).

In [1]:
import pandas as pd

## 1. Load the data

In [2]:
file_path ="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UDKAZw-kz18Yj8P6icf_qw/survey-data-duplicates.csv"

# Load the dataset into a DataFrame
df = pd.read_csv(file_path)

# Display the first few rows
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

## 2. Identify duplicate rows

In [3]:
# 1. Count duplicate rows
dup_count = df.duplicated().sum()
print("Number of duplicate rows:", dup_count)

# 2. Show the first few duplicate rows
print(df[df.duplicated()].head())

Number of duplicate rows: 10       ResponseId                      MainBranch                 Age  \
65437           1  I am a developer by profession  Under 18 years old   
65438           2  I am a developer by profession     35-44 years old   
65439           3  I am a developer by profession     45-54 years old   
65440           4           I am learning to code     18-24 years old   
65441           5  I am a developer by profession     18-24 years old   

                Employment RemoteWork   Check  \
65437  Employed, full-time     Remote  Apples   
65438  Employed, full-time     Remote  Apples   
65439  Employed, full-time     Remote  Apples   
65440   Student, full-time        NaN  Apples   
65441   Student, full-time        NaN  Apples   

                                        CodingActivities  \
65437                                              Hobby   
65438  Hobby;Contribute to open-source projects;Other...   
65439  Hobby;Contribute to open-source projects;Other...  

## 3. Remove duplicates and verify

In [4]:
# 1. Remove duplicates
df = df.drop_duplicates()

# 2. Verify the removal
print("Duplicate rows after removal:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicate rows after removal: 0
New shape: (65437, 114)

## 4. Handle missing values
Count missing values in every column. `EdLevel` is a categorical column, so I filled its gaps with the most frequent value.

In [5]:
# 1. Count missing values in every column
print(df.isnull().sum())

# 2. Fill missing EdLevel values with the most frequent value
most_frequent = df['EdLevel'].mode()[0]
print("Most frequent EdLevel:", most_frequent)

df['EdLevel'] = df['EdLevel'].fillna(most_frequent)

# Verify
print("Missing EdLevel after imputing:", df['EdLevel'].isnull().sum())

ResponseId                 0
MainBranch                 0
Age                        0
Employment                 0
RemoteWork             10635
                       ...  
JobSatPoints_11        36001
SurveyLength            9257
SurveyEase              9201
ConvertedCompYearly    42012
JobSat                 36321
Length: 114, dtype: int64
Most frequent EdLevel: Bachelor’s degree (B.A., B.S., B.Eng., etc.)
Missing EdLevel after imputing: 0

## 5. Prepare compensation data
`ConvertedCompYearly` already holds annual compensation converted to a common currency, so no further normalisation is needed.

About 64% of respondents did not report compensation. Filling that many gaps with the median would invent pay data for most rows and squash the real spread, so I left the missing values as they are and use only respondents who reported pay (`comp_df`) for any salary analysis.

In [6]:
# 1. Look at the column
print(df['ConvertedCompYearly'].describe())

# 2. Check for missing values
missing = df['ConvertedCompYearly'].isnull().sum()
print("Missing values:", missing)
print(f"Share missing: {missing / len(df):.1%}")

# 3. Keep the missing values as NaN and analyse only respondents who reported pay
comp_df = df.dropna(subset=['ConvertedCompYearly'])
print("Respondents with reported compensation:", len(comp_df))

## 6. Final check

In [7]:
print("Final shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print("Missing EdLevel:", df['EdLevel'].isnull().sum())
print("Missing ConvertedCompYearly (kept as NaN):", df['ConvertedCompYearly'].isnull().sum())
print("Respondents with reported compensation:", len(comp_df))